# Student Health Risk Data Read

This notebook loads the Kaggle train, test, and sample submission files, then creates a stratified train/validation split from the labeled train data.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

## Load Data

In [2]:
ID_COL = "id"
TARGET_COL = "health_condition"
VALIDATION_SIZE = 0.20
RANDOM_STATE = 42

train_full = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
sample_submission = pd.read_csv("data/sample_submission.csv")

In [3]:
print("full train shape:", train_full.shape)
print("test shape:", test.shape)
print("sample submission shape:", sample_submission.shape)

full train shape: (690088, 15)
test shape: (295753, 14)
sample submission shape: (295753, 2)


## Stratified Train/Validation Split

The real test file has no target labels, so validation must come from the labeled train file. Stratification keeps the `health_condition` class proportions nearly identical in train and validation.

In [4]:
train, val = train_test_split(
    train_full,
    test_size=VALIDATION_SIZE,
    stratify=train_full[TARGET_COL],
    random_state=RANDOM_STATE,
)

train = train.reset_index(drop=True)
val = val.reset_index(drop=True)

print("train split shape:", train.shape)
print("validation split shape:", val.shape)
print("test shape:", test.shape)

train split shape: (552070, 15)
validation split shape: (138018, 15)
test shape: (295753, 14)


In [5]:
split_summary = pd.DataFrame([
    {"dataset": "train_full", "rows": len(train_full), "columns": train_full.shape[1]},
    {"dataset": "train_split", "rows": len(train), "columns": train.shape[1]},
    {"dataset": "val_split", "rows": len(val), "columns": val.shape[1]},
    {"dataset": "test", "rows": len(test), "columns": test.shape[1]},
])

split_summary

,dataset,rows,columns
0,train_full,690088,15
1,train_split,552070,15
2,val_split,138018,15
3,test,295753,14


In [6]:
target_distribution = pd.concat(
    {
        "train_full": train_full[TARGET_COL].value_counts(normalize=True).mul(100),
        "train_split": train[TARGET_COL].value_counts(normalize=True).mul(100),
        "val_split": val[TARGET_COL].value_counts(normalize=True).mul(100),
    },
    axis=1,
).round(2)

target_distribution

,train_full,train_split,val_split
at-risk,85.87,85.87,85.87
unhealthy,8.36,8.36,8.36
fit,5.77,5.77,5.77


## Save Split Files

These files should be used for leakage-safe modeling experiments. Keep `data/train.csv` unchanged as the original full labeled dataset.

In [7]:
train_split_path = "data/train_split.csv"
val_split_path = "data/val_split.csv"

train.to_csv(train_split_path, index=False)
val.to_csv(val_split_path, index=False)

print("saved:", train_split_path, train.shape)
print("saved:", val_split_path, val.shape)

saved: data/train_split.csv (552070, 15)
saved: data/val_split.csv (138018, 15)


## Preview

In [8]:
train.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,313415,at-risk,7.17,91.3,26.86,2635.0,1398.0,49.1,2.03,non-veg,medium,average,sedentary,yes,NaN
1,3515,at-risk,NaN,75.1,24.23,2382.0,13466.0,50.8,1.83,balanced,low,average,active,occasional,male
2,501194,at-risk,8.66,82.4,21.41,2314.0,9473.0,23.3,3.09,balanced,low,poor,sedentary,yes,female
3,303602,at-risk,NaN,74.5,22.59,2165.0,7052.0,21.6,1.92,veg,NaN,average,sedentary,occasional,male
4,117943,at-risk,8.83,68.2,22.01,2108.0,13521.0,52.9,2.35,non-veg,medium,average,active,NaN,male


In [9]:
val.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,304516,at-risk,6.82,67.9,27.17,2464.0,13456.0,39.9,2.05,veg,medium,good,moderate,no,other
1,165358,at-risk,7.97,92.9,16.86,NaN,7456.0,26.9,2.14,balanced,high,good,moderate,no,female
2,671841,at-risk,NaN,NaN,22.18,2352.0,4140.0,19.7,2.33,non-veg,low,poor,sedentary,occasional,male
3,403460,at-risk,NaN,83.1,21.93,2620.0,11656.0,41.1,1.22,non-veg,medium,good,moderate,occasional,other
4,351133,at-risk,8.14,80.0,23.41,2442.0,4484.0,38.9,2.32,veg,medium,NaN,sedentary,no,other


In [10]:
test.head()

,id,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,690088,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,veg,high,poor,active,occasional,male
1,690089,NaN,83.1,22.42,1773.0,6801.0,24.5,2.40,balanced,high,poor,sedentary,yes,other
2,690090,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,balanced,medium,poor,active,no,NaN
3,690091,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,veg,low,good,moderate,yes,other
4,690092,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,veg,high,average,active,occasional,other


## Column Summary

In [11]:
train.describe()

,id,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake
count,552070.000000,491368.000000,545874.000000,540984.000000,509948.000000,540949.000000,546559.000000,517205.000000
mean,345126.138754,6.993130,75.103110,22.983562,2226.129001,8614.062522,38.759024,2.188574
std,199238.803823,1.215215,8.179577,2.481363,347.626648,3930.596787,14.740980,0.518771
min,0.000000,3.000000,50.000000,16.000000,1201.000000,1002.000000,0.000000,0.500000
25%,172664.500000,6.160000,69.400000,21.320000,2052.000000,5389.000000,29.200000,1.850000
50%,345184.500000,6.990000,75.100000,22.990000,2240.000000,8856.000000,39.400000,2.170000
75%,517501.750000,7.810000,80.700000,24.660000,2457.000000,12114.000000,49.400000,2.500000
max,690087.000000,10.000000,107.700000,34.820000,3580.000000,14999.000000,99.800000,4.720000


In [12]:
test.describe()

,id,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake
count,295753.000000,263182.000000,292396.000000,289797.000000,273101.000000,289789.000000,292795.000000,277120.000000
mean,837964.000000,6.993517,75.079560,22.984661,2225.514941,8626.630524,38.796712,2.191508
std,85376.681419,1.216369,8.115118,2.478708,347.816761,3924.646677,14.711180,0.518771
min,690088.000000,3.000000,50.000000,16.000000,1201.000000,1002.000000,0.000000,0.500000
25%,764026.000000,6.160000,69.400000,21.320000,2052.000000,5394.000000,29.200000,1.850000
50%,837964.000000,6.990000,75.100000,22.990000,2240.000000,8857.000000,39.400000,2.180000
75%,911902.000000,7.810000,80.600000,24.650000,2455.000000,12120.000000,49.400000,2.500000
max,985840.000000,10.000000,103.700000,34.820000,3574.000000,14999.000000,98.400000,4.720000


In [13]:
nunique_summary = pd.DataFrame({
    "train_full": train_full.nunique(dropna=True),
    "train_split": train.nunique(dropna=True),
    "val_split": val.nunique(dropna=True),
    "test": test.nunique(dropna=True),
})

nunique_summary

,train_full,train_split,val_split,test
bmi,1596,1585,1495,1548.0
calorie_expenditure,2101,2092,1999,2068.0
diet_type,3,3,3,3.0
exercise_duration,856,848,803,817.0
gender,3,3,3,3.0
health_condition,3,3,3,NaN
heart_rate,537,534,515,526.0
id,690088,552070,138018,295753.0
physical_activity_level,3,3,3,3.0
sleep_duration,701,699,688,692.0


In [14]:
dtype_summary = pd.DataFrame({
    "train_full_dtype": train_full.dtypes.astype(str),
    "test_dtype": test.dtypes.astype(str),
})

dtype_summary

,train_full_dtype,test_dtype
bmi,float64,float64
calorie_expenditure,float64,float64
diet_type,object,object
exercise_duration,float64,float64
gender,object,object
health_condition,object,NaN
heart_rate,float64,float64
id,int64,int64
physical_activity_level,object,object
sleep_duration,float64,float64


## Missing Values

In [15]:
missing_count_summary = pd.DataFrame({
    "train_full": train_full.isna().sum(),
    "train_split": train.isna().sum(),
    "val_split": val.isna().sum(),
    "test": test.isna().sum(),
})

missing_count_summary

,train_full,train_split,val_split,test
bmi,13898,11086,2812,5956.0
calorie_expenditure,52853,42122,10731,22652.0
diet_type,6901,5532,1369,2958.0
exercise_duration,6901,5511,1390,2958.0
gender,21373,17100,4273,9160.0
health_condition,0,0,0,NaN
heart_rate,7833,6196,1637,3357.0
id,0,0,0,0.0
physical_activity_level,36621,29253,7368,15695.0
sleep_duration,75999,60702,15297,32571.0


In [16]:
missing_pct_summary = pd.DataFrame({
    "train_full": train_full.isna().mean().mul(100),
    "train_split": train.isna().mean().mul(100),
    "val_split": val.isna().mean().mul(100),
    "test": test.isna().mean().mul(100),
}).round(2)

missing_pct_summary

,train_full,train_split,val_split,test
bmi,2.01,2.01,2.04,2.01
calorie_expenditure,7.66,7.63,7.78,7.66
diet_type,1.00,1.00,0.99,1.00
exercise_duration,1.00,1.00,1.01,1.00
gender,3.10,3.10,3.10,3.10
health_condition,0.00,0.00,0.00,NaN
heart_rate,1.14,1.12,1.19,1.14
id,0.00,0.00,0.00,0.00
physical_activity_level,5.31,5.30,5.34,5.31
sleep_duration,11.01,11.00,11.08,11.01


## Target Distribution

In [17]:
target_count_summary = pd.DataFrame({
    "train_full": train_full[TARGET_COL].value_counts(),
    "train_split": train[TARGET_COL].value_counts(),
    "val_split": val[TARGET_COL].value_counts(),
})

target_count_summary

,train_full,train_split,val_split
at-risk,592561,474049,118512
unhealthy,57724,46179,11545
fit,39803,31842,7961


In [18]:
target_pct_summary = pd.DataFrame({
    "train_full_pct": train_full[TARGET_COL].value_counts(normalize=True).mul(100),
    "train_split_pct": train[TARGET_COL].value_counts(normalize=True).mul(100),
    "val_split_pct": val[TARGET_COL].value_counts(normalize=True).mul(100),
}).round(2)

target_pct_summary

,train_full_pct,train_split_pct,val_split_pct
at-risk,85.87,85.87,85.87
unhealthy,8.36,8.36,8.36
fit,5.77,5.77,5.77
